# taste-like: Hybrid Batch Embedding

Marqo-FashionSigLIP (768-dim) **hybrid** embeddings: `image * 0.7 + text * 0.3`

Re-embeds ALL products with combined image + product name embeddings.

**Prerequisites:**
- Colab runtime: T4 GPU (Runtime > Change runtime type > T4 GPU)
- Supabase URL and Service Role Key

In [ ]:
# Cell 1: Install dependencies
!pip install -q open_clip_torch supabase Pillow requests tqdm sentencepiece

In [ ]:
# Cell 2: Configuration
# Option A: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar and add:
#   SUPABASE_URL = your project URL
#   SUPABASE_SERVICE_ROLE_KEY = your service role key

try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
except Exception:
    # Option B: Paste directly (delete before sharing notebook)
    SUPABASE_URL = ''  # https://xxxx.supabase.co
    SUPABASE_KEY = ''  # your service_role key

assert SUPABASE_URL and SUPABASE_KEY, 'Set SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY'

DB_BATCH_SIZE = 50  # rows per Supabase update

print(f'Supabase: {SUPABASE_URL[:40]}...')

In [ ]:
# Cell 3: Load Marqo-FashionSigLIP model + tokenizer
import torch
import open_clip

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'hf-hub:Marqo/marqo-fashionSigLIP'
)
model = model.to(device)
tokenizer = open_clip.get_tokenizer('hf-hub:Marqo/marqo-fashionSigLIP')

IMAGE_WEIGHT = 0.7
TEXT_WEIGHT = 0.3

print(f'Model loaded on {device}')
print(f'Hybrid weights: image={IMAGE_WEIGHT}, text={TEXT_WEIGHT}')

# Verify embedding dim
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    dim = model.encode_image(dummy).shape[-1]
print(f'Embedding dim: {dim}')

In [ ]:
# Cell 4: Fetch products to embed
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

all_products = []
offset = 0
page_size = 1000

while True:
    resp = sb.table('products') \
        .select('id, name, fashion_description, image_url') \
        .eq('is_available', True) \
        .is_('embedding', 'null') \
        .range(offset, offset + page_size - 1) \
        .execute()
    batch = resp.data
    all_products.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

no_desc = [p for p in all_products if not p.get('fashion_description')]
print(f'Products to embed (missing embedding): {len(all_products)}')
print(f'Products without fashion_description (name fallback): {len(no_desc)}')

In [ ]:
# Cell 5: Generate hybrid embeddings and update Supabase
# image * 0.7 + text * 0.3, normalized
# GPU batch inference (32) + parallel image download (8 threads)
# Text: fashion_description (rich) > name (fallback)

import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import numpy as np
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

GPU_BATCH_SIZE = 32
DOWNLOAD_WORKERS = 8

def download_image(url, timeout=10):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/131.0.0.0 Safari/537.36',
        'Accept': 'image/webp,image/apng,image/*,*/*;q=0.8',
    }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert('RGB')


def download_batch(products):
    results = []
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {
            pool.submit(download_image, p['image_url']): p
            for p in products
        }
        for future in as_completed(futures):
            product = futures[future]
            try:
                img = future.result()
                results.append((product, img, None))
            except Exception as e:
                results.append((product, None, str(e)))
    return results


failed = []
total_embedded = 0
hybrid_count = 0
image_only_count = 0

for i in tqdm(range(0, len(all_products), GPU_BATCH_SIZE), desc='Batches'):
    batch_products = all_products[i:i + GPU_BATCH_SIZE]

    # Step 1: Download images in parallel
    downloaded = download_batch(batch_products)

    valid = [(p, img) for p, img, err in downloaded if img is not None]
    for p, _, err in downloaded:
        if err is not None:
            failed.append({'id': p['id'], 'url': p['image_url'], 'error': err})

    if not valid:
        continue

    # Step 2: Image embeddings (GPU batch)
    tensors = torch.stack([preprocess_val(img) for _, img in valid]).to(device)
    with torch.no_grad():
        img_embs = model.encode_image(tensors)
        img_embs = img_embs / img_embs.norm(dim=-1, keepdim=True)

        # Step 3: Text embeddings (fashion_description > name fallback)
        names = [p.get('fashion_description') or p.get('name', '') or '' for p, _ in valid]
        has_text = [bool(n.strip()) for n in names]

        if any(has_text):
            text_inputs = [n.strip() if h else 'product' for n, h in zip(names, has_text)]
            tokens = tokenizer(text_inputs).to(device)
            text_embs = model.encode_text(tokens)
            text_embs = text_embs / text_embs.norm(dim=-1, keepdim=True)

            # Step 4: Combine (only for products with text)
            combined = img_embs.clone()
            for j in range(len(valid)):
                if has_text[j]:
                    combined[j] = img_embs[j] * IMAGE_WEIGHT + text_embs[j] * TEXT_WEIGHT
                    hybrid_count += 1
                else:
                    image_only_count += 1
            combined = combined / combined.norm(dim=-1, keepdim=True)
            embeddings_list = combined.cpu().numpy().tolist()
        else:
            embeddings_list = img_embs.cpu().numpy().tolist()
            image_only_count += len(valid)

    # Step 5: Update Supabase
    for (product, _), emb in zip(valid, embeddings_list):
        sb.table('products').update({'embedding': emb}).eq('id', product['id']).execute()

    total_embedded += len(valid)
    time.sleep(0.3)

print(f'\nDone: {total_embedded} embedded ({hybrid_count} hybrid, {image_only_count} image-only)')
print(f'Failed: {len(failed)}')
if failed:
    print('\nFailed products (first 10):')
    for f in failed[:10]:
        print(f'  {f["id"]}: {f["error"]}')

In [ ]:
# Cell 6: Verification — compare hybrid vs image-only
import numpy as np
import json

def parse_embedding(emb):
    if isinstance(emb, str):
        return np.array(json.loads(emb), dtype=np.float32)
    return np.array(emb, dtype=np.float32)

# Count stats
resp = sb.table('products').select('id', count='exact').not_.is_('embedding', 'null').execute()
resp2 = sb.table('products').select('id', count='exact').is_('embedding', 'null').execute()
print(f'Products with embedding: {resp.count}')
print(f'Products without embedding: {resp2.count}')

# Similarity test: same subcategory pair vs different subcategory pair
print('\n--- Subcategory quality check (bags) ---')
tote = sb.table('products').select('id, brand, name, subcategory, embedding') \
    .eq('category', 'bags').eq('subcategory', 'tote') \
    .not_.is_('embedding', 'null').limit(1).execute()
backpack = sb.table('products').select('id, brand, name, subcategory, embedding') \
    .eq('category', 'bags').eq('subcategory', 'backpack') \
    .not_.is_('embedding', 'null').limit(1).execute()
tote2 = sb.table('products').select('id, brand, name, subcategory, embedding') \
    .eq('category', 'bags').eq('subcategory', 'tote') \
    .not_.is_('embedding', 'null').limit(1).offset(1).execute()

if tote.data and backpack.data and tote2.data:
    e_tote = parse_embedding(tote.data[0]['embedding'])
    e_back = parse_embedding(backpack.data[0]['embedding'])
    e_tote2 = parse_embedding(tote2.data[0]['embedding'])

    sim_same = np.dot(e_tote, e_tote2) / (np.linalg.norm(e_tote) * np.linalg.norm(e_tote2))
    sim_diff = np.dot(e_tote, e_back) / (np.linalg.norm(e_tote) * np.linalg.norm(e_back))

    print(f'Tote 1: {tote.data[0]["brand"]} | {tote.data[0]["name"]}')
    print(f'Tote 2: {tote2.data[0]["brand"]} | {tote2.data[0]["name"]}')
    print(f'Backpack: {backpack.data[0]["brand"]} | {backpack.data[0]["name"]}')
    print(f'\nTote↔Tote similarity: {sim_same:.4f}')
    print(f'Tote↔Backpack similarity: {sim_diff:.4f}')
    print(f'Gap (higher = better subcategory separation): {sim_same - sim_diff:.4f}')

In [ ]:
# Cell 7: Test match_products RPC (vector search)
# Pick a product and find similar ones via pgvector

test_product = sb.table('products') \
    .select('id, brand, name, category, price, embedding') \
    .not_.is_('embedding', 'null') \
    .eq('category', 'shoes') \
    .limit(1) \
    .execute()

if test_product.data:
    tp = test_product.data[0]
    print(f'Query: {tp["brand"]} | {tp["name"]} | {tp["category"]} | W{tp["price"]:,}')
    print('-' * 60)

    results = sb.rpc('match_products', {
        'query_embedding': tp['embedding'],
        'query_category': tp['category'],
        'match_threshold': 0.3,
        'match_count': 5
    }).execute()

    for r in results.data:
        print(f'  {r["similarity"]:.4f} | {r["brand"]} | {r["name"]} | W{r["price"]:,}')
else:
    print('No products with embeddings in shoes category')